# YouTube-to-QA: Turning Video Content into Searchable Knowledge

## Problem
YouTube is one of the largest repositories of knowledge on the internet. However, extracting specific information from videos is time-consuming and inefficient — users have to watch the entire video to find answers.

## Solution using GenAI
In this notebook, we use a Retrieval-Augmented Generation (RAG) pipeline with GenAI tools to transform a YouTube video into a searchable knowledge base. A user can ask a natural language question, and the system will return an accurate answer derived from the video transcript.

## GenAI Capabilities Used
- **Embeddings**: Convert transcript into dense vector form
- **Vector search (ChromaDB)**: Store and search transcript segments
- **Retrieval-Augmented Generation (RAG)**: Retrieve relevant info and generate a grounded answer

This notebook demonstrates how GenAI can turn passive video content into an interactive, intelligent experience.


<h2>Installations</h2>

In [1]:
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"
!pip install youtube_transcript_api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 86.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 

<h2> Extracting Transcript from YouTube</h2>
This code takes a YouTube URL and fetches its auto-generated subtitles using the "youtube_transcript_api". It defaults to English, prints the transcript, and stores it for further processing.



In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import NoTranscriptFound

# Can also ask user to enter a YouTube video URL
# video_url = input("Enter the youtube video url.")
video_url = 'https://www.youtube.com/watch?v=pTB0EiLXUC8'

# Extract video ID from URL by splitting at 'v='
video_id = video_url.split('v=')[1]

try:
    # Initialize YouTubeTranscriptApi (default is English)
    ytt_api = YouTubeTranscriptApi()
    
    # Fetch transcript in English
    transcript_text = ytt_api.fetch(video_id ,languages=['en'])

    # Print each line of the transcript
    for snippet in transcript_text:
        print(snippet.text)
        
    # Get the last snippet and total count (optional usage)
    last_snippet = transcript_text[-1]
    snippet_count = len(transcript_text)
    
except NoTranscriptFound:
    print("No transcript available for this language")


[Music]
a popular interview question concerns
the four core concepts in
object-oriented programming this
concepts are encapsulation abstraction
inheritance and polymorphism let's look
at each of these concepts before
object-oriented programming we had
procedure of programming that divided a
program into a set of functions so we
have data stored in a bunch of variables
and functions that operate on the data
this style of programming is very simple
and straightforward often it's what you
learn as part of your first programming
subject at a university but as your
programs grow it will end up with a
bunch of functions that are all over the
place you might find yourself copying
and pasting lines of code over and over
you make a change to one function and
then several other functions break
that's what we call spaghetti code there
is so much interdependence e between all
these functions it becomes problematic
object-oriented programming came to
solve this problem in object-oriented
programmin

<h2>Chunking the Transcript</h2>
Long transcripts are broken down into manageable pieces to avoid model input limits and improve embedding granularity.
Each chunk is roughly 500 characters.

In [3]:
# Function to split a long transcript into smaller chunks
def chunk_transcript(transcript, chunk_size=500):
    chunks = []
    current_chunk = ""
    
    for item in transcript:
        text = item.text
        # If the current chunk is small enough, add more text
        if len(current_chunk) + len(text) <= chunk_size:
            current_chunk += " " + text
        else:
            # Otherwise, finalize the current chunk and start a new one
            chunks.append(current_chunk.strip())
            current_chunk = text
    
    # Add any remaining chunk
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# Chunk the transcript into parts (default size 500 characters)
chunks = chunk_transcript(transcript_text)
print(f"Chunked into {len(chunks)} segments.")
print(chunks[0])  # Preview the first chunk

Chunked into 14 segments.
[Music] a popular interview question concerns the four core concepts in object-oriented programming this concepts are encapsulation abstraction inheritance and polymorphism let's look at each of these concepts before object-oriented programming we had procedure of programming that divided a program into a set of functions so we have data stored in a bunch of variables and functions that operate on the data this style of programming is very simple and straightforward often it's what you


 <h2>Securely Accessing Google API Key</h2>
 Loads your Gemini API key securely from Kaggle Secrets.

In [4]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

<h2>Embedding Transcript and Storing in ChromaDB</h2>
Gemini converts text chunks into embeddings. ChromaDB is used as a vector store to store these embeddings. This enables fast semantic search for later querying.

In [5]:
import chromadb
from google import genai

# Initialize Google and ChromaDB clients
client = genai.Client(api_key=GOOGLE_API_KEY)
chroma_client = chromadb.Client()

# Ask user for a collection name to group embeddings
collection_name = input("Enter the name of the Chroma collection: ")

# List existing collections in Chroma
existing_names = chroma_client.list_collections()

# Create or fetch the specified collection
if collection_name in existing_names:
    collection = chroma_client.get_collection(collection_name)
    print(f"Using existing collection '{collection_name}'.")
else:
    collection = chroma_client.create_collection(collection_name)
    print(f"Created new collection '{collection_name}'.")

# Re-chunk transcript (if needed)
chunks = chunk_transcript(transcript_text)  
metadata = [{'chunk': chunk} for chunk in chunks]

# Generate embeddings using Gemini embedding model
embedding_response = client.models.embed_content(
    model="models/text-embedding-004",
    contents=chunks
)

# Extract embeddings
embeddings = [e.values for e in embedding_response.embeddings]

# Store embeddings in ChromaDB collection
for emb, meta in zip(embeddings, metadata):
    collection.add(
        ids=[str(meta['chunk'])],       # Use chunk text as unique ID
        embeddings=[emb],
        metadatas=meta,                 # Save metadata with the chunk
        documents=[meta['chunk']]       # Store actual chunk text
    )

print(f"Inserted {len(chunks)} chunks into collection '{collection_name}'.")


Enter the name of the Chroma collection:  youtube


Created new collection 'youtube'.
Inserted 14 chunks into collection 'youtube'.


<h2>Finding Relevant Chunks for a Query</h2>
A user query is turned into an embedding. ChromaDB returns most similar transcript chunks based on the query.



In [6]:
from google import genai

# Function to retrieve relevant transcript chunks based on query
def get_relevant_chunks(query, collection, num_results=3):
    # Convert the query into an embedding using Gemini
    response = client.models.embed_content(
        model="models/text-embedding-004",
        contents=[query]
    )
    query_embedding = response.embeddings[0].values

    # Search for similar chunks in ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=num_results
    )

    # Extract document text from results
    if 'documents' in results and results['documents']:
        relevant_chunks = [result[0] for result in results['documents']]
        return relevant_chunks
    else:
        print("No relevant documents found in Chroma.")
        return []


<h2>Generating Final Answer with Gemini</h2>
Chosen chunks are passed into Gemini with the query. The model gives a natural language answer using the transcript as context. This completes the RAG (retrieval-augmented generation) pipeline.



In [7]:
# Function to generate answer from relevant transcript chunks
def generate_answer(query, relevant_chunks):
    # Combine chunks into a single context string
    context = " ".join(relevant_chunks)

    # Construct the prompt for the LLM
    prompt = f"Question: {query}\n\nContext: {context}"

    # Use Gemini to generate an answer
    response = client.models.generate_content(
        model="gemini-2.0-flash", 
        contents=prompt  
    )

    # Extract text from the response
    answer = response.candidates[0].content.parts[0].text
    return answer


<h2>User Query Execution</h2>
Takes a query as user input. Uses the RAG pipeline to return an LLM-generated answer grounded in the transcript.

In [9]:
# Take user query as an input
# user_query = input("Enter your query. e.g: ")
user_query = 'what problem did object oriented programming came to solve?'

# Step 1: Retrieve matching chunks
relevant_chunks = get_relevant_chunks(user_query, collection)

# Step 2: Generate answer from the matched chunks
if relevant_chunks:
    answer = generate_answer(user_query, relevant_chunks)
    print("Answer:", answer)
else:
    print("No relevant chunks found to generate an answer.")


Answer: Object-oriented programming (OOP) came to solve the problem of **complexity and maintainability in large software projects**, specifically addressing the issues associated with what's often called "spaghetti code."

Here's a breakdown of the problems OOP aimed to solve, based on your context:

*   **Spaghetti Code:** As programs grow with procedural or functional approaches, they often become a tangled mess of functions and global data. The code is hard to understand, debug, and modify due to numerous interdependencies.

*   **Code Duplication:** You often find yourself copying and pasting the same or similar code in different parts of the program, leading to redundancy.

*   **Fragility:** Changes in one part of the code can unintentionally break other unrelated parts, due to the tight coupling of functions and data.

*   **Lack of Modularity:** It's difficult to break the program into independent, reusable components.

In essence, OOP aims to:

*   **Organize code** into reus